# Pathwiser 03: trained baselines

Compares the results of the trained models written by `pathwiser baselines run` (`reports/baselines/v1/*.json`) with the training-free references. **Computes nothing.**

| Baseline | What is learned | Training data |
|---|---|---|
| `tfidf_lr` | Logistic regression over TF-IDF / word similarity and relative (listwise) features | the whole train split |
| `st_lr` | LR over pair features of frozen MiniLM embeddings (cosine, m·e, \|m−e\|) | 30,000 weighted train examples |
| `st_lgbm` | LightGBM over the same features | the same 30,000 examples |
| `cross_encoder` | MiniLM cross-encoder fine-tuning (listwise softmax + "none" logit) | 8,000 weighted train examples, 1 epoch (**CPU budget**) |

**Protocol** (`docs/evaluation-protocol.md`): training on train only, hyper-parameters fixed before looking at results (no search), τ tuned on validation,
test read only once in the final run. SGD-X twins are reported separately. A model with a limited training budget is a **lower bound**, not a verdict on the method.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd().resolve()
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "pathwiser").is_dir():
            return p
    raise FileNotFoundError("repo root not found")


ROOT = find_root()
sys.path.insert(0, str(ROOT))
REPORTS = ROOT / "reports" / "baselines" / "v1"

TRAINED = ["tfidf_lr", "st_lr", "st_lgbm", "cross_encoder"]
REFS = ["random", "lexical", "st_message"]               # training-free references
ORDER = REFS + TRAINED
RES = {n: json.loads((REPORTS / f"{n}.json").read_text(encoding="utf-8")) for n in ORDER if (REPORTS / f"{n}.json").exists()}
HAS_TEST = all("test" in r["splits"] for r in RES.values())
SHOWN = "test" if HAS_TEST else "val"

PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#6250d6", "#008300", "#e34948"]
COLOR = {"random": "#9a9893", "lexical": PAL[0], "st_message": PAL[5], "tfidf_lr": PAL[3], "st_lr": PAL[2],
         "st_lgbm": PAL[1], "cross_encoder": PAL[7]}
INK, INK2 = "#0b0b0b", "#52514e"
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.titlesize": 10, "axes.titleweight": "bold",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.alpha": 0.25, "axes.axisbelow": True,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "legend.frameon": False,
})
d = next(iter(RES.values()))["data"]
print("baselines:", list(RES), "| shown split:", SHOWN, "| test sha:", d["test_sha256"][:12])
missing = [n for n in ORDER if n not in RES]
print("missing result files:", missing or "none")

## 1. Headline results

In [ ]:
HEAD = [("top1", "Top-1"), ("top3", "Top-3"), ("mrr", "MRR"), ("decision_accuracy", "Decision acc."),
        ("no_match_precision", "no_match P"), ("no_match_recall", "no_match R"), ("false_routing_rate", "False routing")]


def headline(split):
    return pd.DataFrame({n: {h: r["splits"][split]["overall"][k] for k, h in HEAD} | {"ms/example": r["latency_ms_per_example"][split]}
                         for n, r in RES.items() if split in r["splits"]}).T


print("== Validation (τ tuned on this split, optimistic) ==")
display(headline("val").round(3))
if HAS_TEST:
    print("== Test (τ from validation), honest numbers ==")
    display(headline("test").round(3))

h = headline(SHOWN)
fig, ax = plt.subplots(figsize=(8.5, 3.4))
y = np.arange(len(h)); w = 0.38
for off, col, shade in [(-w / 2, "Top-1", 1.0), (w / 2, "Decision acc.", 0.55)]:
    bars = ax.barh(y + off, h[col], w * 0.92, color=[COLOR[n] for n in h.index], alpha=shade)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in h[col]], fontsize=7, color=INK2, padding=2)
ax.set_yticks(y, h.index); ax.invert_yaxis(); ax.set_xlim(0, 1.05)
ax.set_title(f"Top-1 (dark) and decision accuracy (light), {SHOWN}"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 2. Training cost

In [ ]:
rows = {}
for n in TRAINED:
    if n not in RES:
        continue
    t, p = RES[n].get("training", {}), RES[n]["params"]
    rows[n] = {"init / embedding": p.get("resolved_init") or p.get("embedding_model") or "TF-IDF (train)",
               "train examples": t.get("train_examples"), "pairs": t.get("train_pairs"), "step": t.get("steps"),
               "train min": round(t.get("train_seconds", 0) / 60, 1), "final loss": t.get("final_loss"),
               "features": t.get("n_features")}
display(pd.DataFrame(rows).T)

ce = RES.get("cross_encoder", {}).get("training", {})
if ce.get("loss_curve"):
    cv = pd.DataFrame(ce["loss_curve"])
    fig, ax = plt.subplots(figsize=(8.5, 3.0))
    ax.plot(cv["step"], cv["loss"], color=COLOR["cross_encoder"], lw=2, label="training loss (listwise)")
    ax2 = ax.twiny(); ax2.set_visible(False)
    ax.set_xlabel("step"); ax.set_ylabel("loss")
    for _, r in cv.iloc[::max(1, len(cv) // 6)].iterrows():
        ax.text(r["step"], r["loss"], f"{r['train_acc']:.2f}", fontsize=7, color=INK2, va="bottom")
    ax.set_title("Cross-encoder training: loss (label: in-window training accuracy)")
    plt.tight_layout(); plt.show()
    print("learned 'none' logit:", ce.get("null_logit"), "| pairs/s:", ce.get("pairs_per_second"), "| hardware:", ce.get("hardware"))

## 3. Per dataset

In [ ]:
datasets = sorted(next(iter(RES.values()))["splits"][SHOWN]["datasets"])
print("Top-1:")
display(pd.DataFrame({n: {ds: r["splits"][SHOWN]["datasets"][ds]["top1"] for ds in datasets} for n, r in RES.items()}).T.round(3))
print("Decision accuracy (global τ):")
display(pd.DataFrame({n: {ds: r["splits"][SHOWN]["datasets"][ds]["decision_accuracy"] for ds in datasets} for n, r in RES.items()}).T.round(3))

## 4. Generalisation: seen vs. unseen conditions

For trained models this table is the first real answer to the central question: how much does performance drop on conditions **never seen** in training (`unseen`) and on those of which only an **equivalent** was seen (`equivalent_only`), compared with the case where the same condition was seen (`exact`)?
The slices differ in difficulty, so the comparison has to be made **against the same slice of the training-free reference**.

In [ ]:
SLICES = [("seen_exact:exact", "seen\n(exact)"), ("seen_exact:equivalent_only", "seen\n(equivalent only)"), ("cond:unseen", "unseen")]
sl = pd.DataFrame({n: {lab.replace("\n", " "): r["splits"][SHOWN]["slices"].get(k, {}).get("top1") for k, lab in SLICES} for n, r in RES.items()}).T
print("example counts (route):", {lab.replace("\n", " "): next(iter(RES.values()))["splits"][SHOWN]["slices"].get(k, {}).get("n") for k, lab in SLICES})
display(sl.round(3))

fig, ax = plt.subplots(figsize=(10, 3.8))
names = list(RES); x = np.arange(len(SLICES)); w = 0.8 / len(names)
for i, n in enumerate(names):
    vals = [RES[n]["splits"][SHOWN]["slices"].get(k, {}).get("top1") or 0 for k, _ in SLICES]
    bars = ax.bar(x + (i - (len(names) - 1) / 2) * w, vals, w * 0.9, color=COLOR[n], label=n)
    ax.bar_label(bars, labels=[f"{v:.2f}" for v in vals], fontsize=6.5, color=INK2, padding=1.5, rotation=90)
ax.set_xticks(x, [lab for _, lab in SLICES]); ax.set_ylim(0, 1.15); ax.set_ylabel("Top-1")
ax.set_title(f"Top-1: seen and unseen conditions ({SHOWN})"); ax.legend(ncol=4, fontsize=8, loc="upper left"); ax.grid(axis="x", visible=False)
plt.tight_layout(); plt.show()

OTHER = [g for g in ["ctx:none", "ctx:first_turn", "ctx:later_turn", "svc:seen", "svc:unseen", "k:2-3", "k:4-6", "k:7+"]
         if g in next(iter(RES.values()))["splits"][SHOWN]["slices"]]
print("Top-1, other slices:")
display(pd.DataFrame({n: {g: r["splits"][SHOWN]["slices"][g]["top1"] for g in OTHER} for n, r in RES.items()}).T.round(3))

## 5. Paraphrase robustness (SGD-X twins)

In [ ]:
rows = {}
for n, r in RES.items():
    pc = r["splits"][SHOWN]["paraphrase_consistency"]
    rows[n] = {"Top-1 original": pc["top1_main"], "Top-1 paraphrase": pc["top1_paraphrase"], "same pick": pc["same_selected_candidate"],
               "same abstain decision": pc["same_abstain_decision"]}
pdf = pd.DataFrame(rows).T
display(pdf.round(3))
fig, ax = plt.subplots(figsize=(8.5, 3.2))
for i, n in enumerate(pdf.index):
    a, b = pdf.loc[n, "Top-1 original"], pdf.loc[n, "Top-1 paraphrase"]
    ax.plot([a, b], [i, i], color="#c9c8c2", lw=2, zorder=1)
    ax.scatter([a], [i], s=36, color=COLOR[n], zorder=2)
    ax.scatter([b], [i], s=36, facecolor="white", edgecolor=COLOR[n], lw=1.8, zorder=2)
    ax.text(max(a, b) + 0.015, i, f"{a:.2f} → {b:.2f}", va="center", fontsize=7.5, color=INK2)
ax.set_yticks(range(len(pdf)), pdf.index); ax.invert_yaxis(); ax.set_xlim(0, 1.0); ax.set_xlabel("Top-1")
ax.set_title(f"SGD-X: original (filled) -> held-out paraphrase (hollow), {SHOWN}"); ax.grid(axis="y", visible=False)
plt.tight_layout(); plt.show()

## 6. no_match behaviour

In [ ]:
NM = ["no_match_precision", "no_match_recall", "no_match_f1", "false_routing_rate", "abstain_rate", "false_abstain_rate", "wrong_route_rate"]
display(pd.DataFrame({n: {k: r["splits"][SHOWN]["overall"][k] for k in NM} for n, r in RES.items()}).T.round(3))
kinds = [g for g in ["nm_kind:other_family", "nm_kind:sibling_removed", "nm_kind:oos"] if g in next(iter(RES.values()))["splits"][SHOWN]["slices"]]
print("Abstention by no_match kind (recall):")
display(pd.DataFrame({n: {g: r["splits"][SHOWN]["slices"][g]["no_match_recall"] for g in kinds} for n, r in RES.items()}).T.round(3))

## 7. Findings (test, τ from validation)

All numbers are a single run / single seed; no confidence intervals were computed, so differences of about 1 point should not be treated as distinguishable. No hyper-parameter search was done; test was read only once.

**1. Training helped, but less than expected and unevenly.**
- Top-1: best training-free `st_message` 0.846 -> `st_lr` 0.866, `st_lgbm` 0.875, `cross_encoder` 0.854. So Top-1 gains 1-3 points.
- The gain in decision accuracy is larger: `st_message` 0.725 -> `st_lgbm` 0.783 and `cross_encoder` 0.783 (+5.8 points). Most of the gain comes from learning *when to abstain*, not from picking the right condition.
- `tfidf_lr` (Top-1 0.619, decision 0.584) is practically the same as the word-overlap reference (`lexical` 0.616 / 0.579): word features are not beaten even with training; the real jump comes from embeddings.

**2. Given its budget, the cross-encoder is not the "winner".**
- On Top-1 `st_lgbm` (0.875) and `st_lr` (0.866) are ahead of the cross-encoder (0.854); on decision accuracy it ties with `st_lgbm` (0.783). Its edge is on `no_match`: F1 0.721 (`st_lgbm` 0.660), recall 0.716 (0.616), false routing 0.155 (0.159).
- The cross-encoder trained on only 8,000 examples (28,855 pairs), 1 epoch, 39.5 min on CPU; in the last steps training accuracy fluctuated around 0.80-0.82 and the loss between 0.45 and 0.57, so it cannot be called converged. This is a **lower bound**: "the cross-encoder is worse than embeddings + LightGBM" cannot be concluded, only "not better at this budget".
- The validation -> test drop is largest for the cross-encoder (decision accuracy 0.820 -> 0.783, -3.7 points; `st_lgbm` -2.4, `st_lr` -1.2). Since τ is tuned on validation, this measures the optimism.
- Latency: cross-encoder about 33 ms/example, `st_message` about 2.4 ms. The 0.13 ms of `st_lgbm` uses embeddings from the disk cache and is therefore **not comparable** (`st_lr` on test without cache: 3.84 ms).

**3. A first real hint on the generalisation question (read with care).**
- Top-1, `seen (exact)` -> `unseen`: `st_lgbm` 0.909 -> 0.854, `st_lr` 0.899 -> 0.847, `cross_encoder` 0.902 -> 0.837 (-6.5 points), training-free `st_message` 0.871 -> 0.845. So trained models gain more on seen conditions; on unseen conditions they are close to the training-free reference (`st_lr` 0.847 ≈ `st_message` 0.845 on `unseen`, `st_lgbm` 0.854).
- This **partly** supports the hypothesis that "training memorises conditions": part of the gain from training is specific to seen conditions. It is not conclusive: the slices differ in dataset mix (their difficulty is not the same) and the `exact`/`unseen` n differ (9,179 / 8,924). The comparison was therefore made against the training-free reference on the same slice.
- `equivalent_only` (different service/wording): cross-encoder 0.805 (-9.7 points against exact; the largest drop among the trained embedding/cross models, `tfidf_lr` -16), `st_lgbm` 0.852 (-5.7). At this budget the cross-encoder is the model most sensitive to the *wording* of a condition.
- `svc:unseen` -> `svc:seen`: a 9-14 point gap for embedding-based models (`cross_encoder` 0.799 vs 0.938; 20 points for the word-based `tfidf_lr`); an unseen service is still the biggest difficulty and the clearest generalisation gap.

**4. Paraphrase robustness did not improve with training.**
- Top-1 original -> paraphrase: `st_message` 0.809 -> 0.758 (-5.1), `st_lr` -6.7, `st_lgbm` -6.6, `cross_encoder` 0.837 -> 0.783 (-5.4), `tfidf_lr` -6.7.
- The cross-encoder is best at keeping the same pick (0.738; `st_lgbm` / `st_lr` 0.713, `st_message` 0.691), but on the same abstain decision `st_message` (0.907) and the cross-encoder (0.886) beat the others (about 0.85). No model is stable under paraphrase: changing the condition text changes the pick in roughly 1 of every 3-4 examples (same-pick rate 0.69-0.74, `tfidf_lr` 0.46).

**5. Shared weaknesses (no trained model solved them).**
- `nm_kind:sibling_removed` (the true condition was removed from the candidates, its similar sibling remains): recall 0.39-0.54; best are `tfidf_lr` 0.543 and the cross-encoder 0.496. The model cannot stop routing to the sibling; the riskiest error type (`false routing`) is here.
- `ABCD`: `st_lgbm` 0.684, cross-encoder 0.609 (**below** `st_message` 0.643); decision accuracy of the cross-encoder is also 0.481. The cause was not tested in this analysis; one possible explanation is the context representation (plain concatenation), but this cannot be said without an ablation.
- `ctx:first_turn`: cross-encoder 0.611, `st_message` 0.652; the first dialogue turn is the weakest context kind.
- `nm_kind:oos`: cross-encoder 0.877, `st_message` 0.923; none of the trained models beat the training-free reference on CLINC-style out-of-scope examples (`st_lr` 0.657, `st_lgbm` 0.782).

**What followed:** the decision layer and encoder comparison in `docs/decision-layer.md` (a fusion of two LightGBM scorers is the best system so far).